# First-level (run-level) models

One GLM per run (nilearn), fitted twice: unsmoothed maps for the ROI analyses and 6 mm FWHM maps for the whole-brain analyses.

**Vampire Survivors (VS) runs**:tedana-denoised data, "block" chest model:

| Regressor | Onset | Duration |
|---|---|---|
| `anticipation` | chest opens | until the items are shown |
| `reveal_1`, `reveal_3`, `reveal_5` | items shown | 6 s, split by number of items |
| `boss_kill` | boss dies | 0 (impulse) |


**Loot-box (LB) run**: fMRIPrep data; the three loot-box videos are modelled and the text slides are the baseline.
Minimal parameters set (6 motion parameters, 2 aCompCor), because the run is only 2 minutes.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nibabel as nib
from nilearn.glm.first_level import FirstLevelModel, make_first_level_design_matrix
from nilearn.plotting import plot_design_matrix

ROOT = Path("/srv/lab/fmri/crave")
BIDS = ROOT / "crave_bids"
CODE = BIDS / "code"
FMRIPREP = ROOT / "derivatives" / "fmriprep"
TEDANA = ROOT / "derivatives" / "tedana"
QC = ROOT / "derivatives" / "qc"
FIRSTLEVEL = ROOT / "derivatives" / "firstlevel"
SURVEY = CODE / "CRAVE MRI Post Survey_November 15, 2026_17.02.csv"   # round conditions come from the during scan report

TR = 1.761
SLICE_REF = 0.5          # fMRIPrep slice-timing reference (fraction of a TR)
REVEAL_DUR = 6.0         # seconds
SPACE = "space-MNI152NLin2009cAsym_res-2"
MOTION = ["trans_x", "trans_y", "trans_z", "rot_x", "rot_y", "rot_z"]
VS_REGRESSORS = ["anticipation", "reveal_1", "reveal_3", "reveal_5", "boss_kill"]
LB_REGRESSORS = ["video_lvl1", "video_lvl3", "video_lvl5"]

run_qc = pd.read_csv(QC / "run_qc.csv")
excluded_runs = set(zip(run_qc.loc[run_qc["run_excluded"], "sub"], run_qc.loc[run_qc["run_excluded"], "run_label"]))

## 1. Events files

`build_vs_events.py` pairs each coded gameplay video with its scan run (play order plus the stimulus-PC clock),
sets each round's condition from the post-scan survey (the programmed mod settings), applies the manual
decisions in `run_overrides.csv`, and writes `events.tsv` into BIDS. Video 0:00 = first volume.
`make_lb_events.py` writes the loot-box events from the task timeline.

In [ ]:
!python {CODE}/build_vs_events.py --survey "{SURVEY}" --overrides {CODE}/run_overrides.csv --overwrite
!python {CODE}/make_lb_events.py --overwrite

## 2. Functions

In [ ]:
def vs_events(events_file):
    # chest -> anticipation (block) + reveal by size (6 s); boss death -> impulse
    ev = pd.read_csv(events_file, sep="\t")
    ev["magnitude"] = pd.to_numeric(ev["magnitude"], errors="coerce")
    rows = []
    for _, e in ev.iterrows():
        if e["trial_type"] == "boss_death":
            rows.append([e["onset"], 0.0, "boss_kill"])
        if e["trial_type"] == "lootbox" and e["duration"] > 0 and e["magnitude"] in (1, 3, 5):
            rows.append([e["onset"], e["duration"], "anticipation"])
            rows.append([e["onset"] + e["duration"], REVEAL_DUR, "reveal_%d" % e["magnitude"]])
    out = pd.DataFrame(rows, columns=["onset", "duration", "trial_type"])
    out = out.drop_duplicates(["onset", "trial_type"])      # a few events were coded twice
    return out, ev["condition"].iloc[0]


def lb_events(events_file):
    # keep the three videos; the text slides are the baseline
    ev = pd.read_csv(events_file, sep="\t")
    return ev[ev["trial_type"].isin(LB_REGRESSORS)][["onset", "duration", "trial_type"]]


def nuisance(prefix, sub, minimal=False):
    func = FMRIPREP / sub / "ses-01" / "func"
    conf = pd.read_csv(func / f"{prefix}_desc-confounds_timeseries.tsv", sep="\t")
    if minimal:
        cols = MOTION + ["a_comp_cor_00", "a_comp_cor_01"]
    else:
        cols = [m + s for m in MOTION for s in ["", "_derivative1", "_power2", "_derivative1_power2"]]
        cols = cols + ["a_comp_cor_%02d" % i for i in range(5)]
    cols = cols + [c for c in conf.columns if c.startswith("cosine") or c.startswith("non_steady_state_outlier")]
    X = conf[cols].fillna(0)

    # one spike regressor per censored volume
    censored = pd.read_csv(QC / "censor" / f"{prefix}_desc-censor_timeseries.tsv", sep="\t")["censored"].to_numpy() == 1
    spikes = pd.DataFrame(np.eye(len(X))[:, censored], columns=["spike_%04d" % i for i in np.where(censored)[0]])
    X = pd.concat([X, spikes], axis=1)
    X = X.loc[:, X.std() > 0].T.drop_duplicates().T       # drop constant and duplicate columns
    return X, censored


def usable_events(events, censored):
    # usable = at least half of the window (onset + 4 s to offset + 6 s) is uncensored
    times = (np.arange(len(censored)) + SLICE_REF) * TR
    counts = {}
    for name, g in events.groupby("trial_type"):
        n_ok = 0
        for on, du in zip(g["onset"], g["duration"]):
            window = (times >= on + 4) & (times <= on + du + 6)
            if window.any() and (~censored[window]).mean() >= 0.5:
                n_ok += 1
        counts[name] = (len(g), n_ok)
    return counts


def vif(dm, regressors):
    # variance inflation factor of each task regressor
    out = {}
    for r in regressors:
        y = dm[r].to_numpy()
        X = np.column_stack([np.ones(len(dm)), dm.drop(columns=r).to_numpy()])
        resid = y - X @ np.linalg.lstsq(X, y, rcond=None)[0]
        r2 = 1 - (resid ** 2).sum() / ((y - y.mean()) ** 2).sum()
        out[r] = 1 / (1 - r2)
    return out

## 3. Example design matrix (one VS run)

In [ ]:
sub, prefix = "sub-016202", "sub-016202_ses-01_task-VS_run-01"
events, condition = vs_events(BIDS / sub / "ses-01" / "func" / f"{prefix}_events.tsv")
X, censored = nuisance(prefix, sub)
events = events[events["onset"] < len(X) * TR]          # drop events after the scan ended
frame_times = (np.arange(len(X)) + SLICE_REF) * TR

dm = make_first_level_design_matrix(frame_times, events, hrf_model="spm", drift_model=None,
                                    add_regs=X.to_numpy(), add_reg_names=list(X.columns))
print(condition, "|", dm.shape[1], "columns |", int(censored.sum()), "censored volumes")

plot_design_matrix(dm)
plt.show()

task_cols = [c for c in VS_REGRESSORS if c in dm.columns]
dm[task_cols].plot(subplots=True, figsize=(12, 6), legend=True, xlabel="time (s)")
plt.show()

vif(dm, task_cols)

## 4. Fit all VS runs (unsmoothed and 6 mm)

In [ ]:
def fit_vs_run(events_file, fwhm, out_dir):
    prefix = events_file.name.replace("_events.tsv", "")
    sub = prefix.split("_")[0]
    run_label = prefix.split("task-")[1]
    events, condition = vs_events(events_file)
    X, censored = nuisance(prefix, sub)
    events = events[events["onset"] < len(X) * TR]      # drop events after the scan ended
    frame_times = (np.arange(len(X)) + SLICE_REF) * TR
    dm = make_first_level_design_matrix(frame_times, events, hrf_model="spm", drift_model=None,
                                        add_regs=X.to_numpy(), add_reg_names=list(X.columns))
    present = [c for c in VS_REGRESSORS if c in dm.columns]

    bold = TEDANA / sub / "ses-01" / "func" / f"{prefix}_{SPACE}_desc-denoised_bold.nii.gz"
    mask = FMRIPREP / sub / "ses-01" / "func" / f"{prefix}_{SPACE}_desc-brain_mask.nii.gz"
    model = FirstLevelModel(t_r=TR, slice_time_ref=SLICE_REF, mask_img=str(mask), smoothing_fwhm=fwhm,
                            noise_model="ar1", hrf_model="spm", drift_model=None, minimize_memory=True)
    model.fit(str(bold), design_matrices=[dm])

    # contrasts: each regressor, the average of the reveal sizes, anticipation vs boss kill
    contrasts = {c: c for c in present}
    reveals = [c for c in ["reveal_1", "reveal_3", "reveal_5"] if c in present]
    if len(reveals) > 1:
        contrasts["reveal_all"] = "(" + "+".join(reveals) + ")/%d" % len(reveals)
    if "anticipation" in present and "boss_kill" in present:
        contrasts["anticipation_vs_boss"] = "anticipation-boss_kill"

    (out_dir / sub).mkdir(parents=True, exist_ok=True)
    dm.to_csv(out_dir / sub / f"{prefix}_design.csv")
    for name, formula in contrasts.items():
        for output, tag in [("effect_size", "effect"), ("effect_variance", "variance"), ("z_score", "z")]:
            model.compute_contrast(formula, output_type=output).to_filename(
                str(out_dir / sub / f"{prefix}_contrast-{name}_stat-{tag}.nii.gz"))

    row = {"sub": sub, "run": run_label, "condition": condition, "n_vols": len(X),
           "pct_censored": 100 * censored.mean(), "n_nuisance": X.shape[1]}
    for name, (n_all, n_ok) in usable_events(events, censored).items():
        row["n_" + name] = n_all
        row["usable_" + name] = n_ok
    for name, value in vif(dm, present).items():
        row["vif_" + name] = value
    return row

In [ ]:
vs_files = sorted(BIDS.glob("sub-*/ses-01/func/*_task-VS_run-*_events.tsv"))
vs_files = [f for f in vs_files if (f.name.split("_")[0], f.name.split("task-")[1].replace("_events.tsv", "")) not in excluded_runs]
print(len(vs_files), "VS runs")

for fwhm in [None, 6]:
    out_dir = FIRSTLEVEL / ("VS_bold-tedana_fwhm-%d_chest-block_rev-6" % (fwhm or 0))
    rows = []
    for f in vs_files:
        rows.append(fit_vs_run(f, fwhm, out_dir))
        print(fwhm, f.name)
    pd.DataFrame(rows).to_csv(out_dir / "firstlevel_summary.csv", index=False)

## 5. Fit the loot-box runs

In [ ]:
out_dir = FIRSTLEVEL / "LB_bold-fmriprep_fwhm-6_base-text_nuis-min"
rows = []
for events_file in sorted(BIDS.glob("sub-*/ses-01/func/*_task-LB_events.tsv")):
    prefix = events_file.name.replace("_events.tsv", "")
    sub = prefix.split("_")[0]
    if (sub, "LB") in excluded_runs:
        continue
    events = lb_events(events_file)
    X, censored = nuisance(prefix, sub, minimal=True)
    frame_times = (np.arange(len(X)) + SLICE_REF) * TR
    dm = make_first_level_design_matrix(frame_times, events, hrf_model="spm", drift_model=None,
                                        add_regs=X.to_numpy(), add_reg_names=list(X.columns))

    func = FMRIPREP / sub / "ses-01" / "func"
    model = FirstLevelModel(t_r=TR, slice_time_ref=SLICE_REF, smoothing_fwhm=6, noise_model="ar1",
                            hrf_model="spm", drift_model=None, minimize_memory=True,
                            mask_img=str(func / f"{prefix}_{SPACE}_desc-brain_mask.nii.gz"))
    model.fit(str(func / f"{prefix}_{SPACE}_desc-preproc_bold.nii.gz"), design_matrices=[dm])

    contrasts = {c: c for c in LB_REGRESSORS}
    contrasts["video_all"] = "(video_lvl1+video_lvl3+video_lvl5)/3"
    (out_dir / sub).mkdir(parents=True, exist_ok=True)
    for name, formula in contrasts.items():
        for output, tag in [("effect_size", "effect"), ("effect_variance", "variance"), ("z_score", "z")]:
            model.compute_contrast(formula, output_type=output).to_filename(
                str(out_dir / sub / f"{prefix}_contrast-{name}_stat-{tag}.nii.gz"))
    row = {"sub": sub, "n_vols": len(X), "pct_censored": 100 * censored.mean()}
    row.update({"vif_" + k: v for k, v in vif(dm, LB_REGRESSORS).items()})
    rows.append(row)
    print(prefix)

pd.DataFrame(rows).to_csv(out_dir / "firstlevel_summary.csv", index=False)

## 6. First-level summary table

In [ ]:
summary = pd.read_csv(FIRSTLEVEL / "VS_bold-tedana_fwhm-0_chest-block_rev-6" / "firstlevel_summary.csv")
table = pd.DataFrame({
    "runs": [summary["n_" + r].notna().sum() for r in VS_REGRESSORS],
    "events per run (median)": [summary["n_" + r].median() for r in VS_REGRESSORS],
    "usable events (median)": [summary["usable_" + r].median() for r in VS_REGRESSORS],
    "VIF (median)": [summary["vif_" + r].median() for r in VS_REGRESSORS],
}, index=VS_REGRESSORS)
print(len(summary), "runs | median nuisance regressors:", summary["n_nuisance"].median())
table.round(2)